In [0]:
dbutils.widgets.dropdown(name = 'Environment', defaultValue = 'dev', choices = ['qa','dev','prd'], label = 'Environment')
env = dbutils.widgets.get("Environment")

In [0]:
silverTableName = f"saleslake_{env}.silver_{env}.cleanedsales"
print(silverTableName)

bronzeTableName = f"saleslake_{env}.bronze_{env}.rawsales"
print(bronzeTableName)

srcFileLoc=f"/Volumes/saleslake_{env}/silver_{env}/vol_saleslake_src_files_{env}/daily_sales/"
print(srcFileLoc)



In [0]:
spark.sql(f"""
INSERT INTO {silverTableName} 
SELECT DISTINCT
    CAST(TRIM(sale_id) AS INTEGER) as sale_id ,
    UPPER(TRIM(product)) as product,
    UPPER(TRIM(category)) as category,
    CAST(TRIM(quantity) AS INTEGER) as quantity,
    CAST(TRIM(price) AS DOUBLE) as price,
    TO_DATE(TRIM(sale_date),'yyyy-MM-dd') as sale_date,
    UPPER(TRIM(region)) as region,
    CURRENT_TIMESTAMP() as ingest_ts
FROM {bronzeTableName}
WHERE ingest_ts > (
                    SELECT coalesce(MAX(ingest_ts),TO_DATE('1990-01-01','yyyy-MM-dd')) 
                    FROM {silverTableName}
                    )
ORDER BY CAST(TRIM(sale_id) AS INTEGER)
""")

In [0]:
%sql
--SELECT * FROM saleslake_dev.silver_dev.cleanedsales;